<a href="https://colab.research.google.com/github/degasbr1964/Tech-Challenge-Fase-1/blob/main/Tech_Challenge_Fase_1_Edgar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xlrd

Como melhorar o NPS das vendas online da empresa. Porque o usuário se torna detrator ou promotor da empresa.

In [2]:
# Libs - bibliotecas de exploração e manipulação de dados
import pandas as pd
import numpy as np

# Libs Gráficas
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Avisos
import warnings
warnings.filterwarnings("ignore")

In [4]:
nps = pd.read_csv("/content/sample_data/desafio_nps_fase_1.csv")

In [ ]:
nps.head(20)

# Tamanho da base de dados

In [ ]:
nps.shape

In [ ]:
print(f'A base de dados possui {nps.shape[0]} linhas e {nps.shape[1]} colunas')

Verificar o tipo das colunas

In [ ]:
nps.dtypes

Existem valores nulos?

In [ ]:
nps.isnull().sum().sum()

Não existem valores nulos

In [ ]:
nps.columns

Cria uma base com menos colunas

In [ ]:
df_nps = nps[['customer_age', 'customer_region',
       'customer_tenure_months', 'order_value', 'items_quantity',
       'discount_value', 'payment_installments', 'delivery_time_days',
       'delivery_delay_days', 'freight_value', 'delivery_attempts',
       'customer_service_contacts', 'resolution_time_days', 'nps_score',
       'repeat_purchase_30d', 'complaints_count', 'csat_internal_score']]

In [ ]:
df_nps.head()

##Gerando uma amostra da base inicial

In [ ]:
np.random.seed(42)
amostra = df_nps.sample(n=1000,replace=False)

In [ ]:
amostra.head()

Comparando a amostra com a base de dados original

In [ ]:
nps.describe()

In [ ]:
amostra.describe()

Altera os nomes das colunas para português

In [ ]:
amostra.columns = ['idade', 'regiao', 'tempo_relacionamento','valor_compra','quantidade','desconto', 'parcelas','tempo_entrega', 'dias_atraso','valor_frete', 'tentativas_entrega', 'chamados_sac','tempo_resolucao','nps','recompra_30d','num_reclamacoes', 'csat']

Estatística descritiva

In [ ]:
amostra['nps'].hist()

In [ ]:
amostra.describe().round(2)

In [ ]:
def nps_class(nps):
    if nps <= 6:
        return "Detrator"
    elif nps >= 9:
        return "Promotor"
    else:
        return "Passivo"

Classificação do cliente segundo o NPS
(fonte: Bain & Company)

*   Promotores (9-10)
*   Passivos (7-8)
*   Detratores (0-6)








In [ ]:
amostra.head()

## *Função para classificar NPS*

In [ ]:
amostra["classificacao_nps"] = amostra["nps"].apply(nps_class)


In [ ]:
amostra.head()

Proporção da distribuição da classificação em %

In [ ]:
amostra['classificacao_nps'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
amostra['classificacao_nps'].value_counts()

Percentualmente qual a distribuição dos clientes segundo a classificação NPS?

In [ ]:
nps_class_counts = amostra['classificacao_nps'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(nps_class_counts, labels=nps_class_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))

# Draw a circle at the center to make it a donut chart
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)

ax.set_title('Distribuição de Clientes por NPS')
ax.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

Mais de 70% de detratores indica uma questão a ser investigada


Quantos usuários fizeram uma recompra nos 30 dias posteriores à compra?

In [ ]:
amostra['recompra_30d'].value_counts()

Olhando a taxa de recompra após 30 dias, dependendo da linha de produtos a ser analisada pode até ser aceitável segundo o relatório Prax (https://www.prax.ai/blog/benchmarks-ecommerce?utm_source=copilot.com) mas podemos entender também que 30 dias é um período muito curto de análise para sobre a retenção do cliente.

In [ ]:
nps_grouped_analysis = amostra.groupby('classificacao_nps').mean(numeric_only=True).round(2)
display(nps_grouped_analysis.sort_values(by='classificacao_nps'))

### Análise de médias por classificação de NPS para identificar motivos de NPS baixo

Com a tabela acima, podemos comparar as médias de diversas métricas para cada grupo de NPS. Fatores com valores significativamente maiores para 'Detrator' e menores para 'Promotor' são os potenciais maiores motivadores de um NPS baixo.

Quanto maior os dias de atraso, o tempo de resolução, os chamados ao SAC, o número de reclamações, menor o NPS e consequentemente torna esses clientes detratores da empresa.

In [ ]:
pedidos_atrasados_por_regiao = amostra[amostra['dias_atraso'] > 0].groupby('regiao').size().reset_index(name='pedidos_atrasados')
display(pedidos_atrasados_por_regiao)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='regiao', y='pedidos_atrasados', data=pedidos_atrasados_por_regiao, palette='viridis')
plt.title('Número de Pedidos Atrasados por Região')
plt.xlabel('Região')
plt.ylabel('Pedidos Atrasados')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Como podemos ver, os atrasos de entrega independem da regiaão, portanto não é um problema localizado, e sim da logística como um todo.

In [ ]:
detratores_com_atraso = amostra[(amostra['classificacao_nps'] == 'Detrator') & (amostra['dias_atraso'] > 0)]
num_detratores_com_atraso = detratores_com_atraso.shape[0]
print(f"O número de Detratores que tiveram atraso na entrega é: {num_detratores_com_atraso}")

In [ ]:
detratores_com_reclamacao = amostra[(amostra['classificacao_nps'] == 'Detrator') & (amostra['num_reclamacoes'] > 0)]
num_detratores_com_reclamacao = detratores_com_reclamacao.shape[0]
print(f"O número de Detratores que fizeram reclamação é: {num_detratores_com_reclamacao}")

In [ ]:
total_detratores = amostra[amostra['classificacao_nps'] == 'Detrator'].shape[0]
detratrores_atraso_reclamacao = amostra[(amostra['classificacao_nps'] == 'Detrator') & (amostra['dias_atraso'] > 0) & (amostra['num_reclamacoes'] > 0)]
num_detratores_atraso_reclamacao = detratrores_atraso_reclamacao.shape[0]

if total_detratores > 0:
    porcentagem_detratores_atraso_reclamacao = (num_detratores_atraso_reclamacao / total_detratores) * 100
    print(f"Do total de Detratores, {porcentagem_detratores_atraso_reclamacao:.2f}% tiveram atraso na entrega e fizeram reclamação.")
else:
    print("Não há detratores na base de dados para realizar o cálculo.")

94,73% dos clientes que tiveram atraso na entrega, fizeram reclamação ao SAC, isso nos leva a acreditar que diminuindo os problemas de logística diminuiremos os atendimentos no SAC.

In [ ]:
detratores_reclamacao_sem_atraso = amostra[(amostra['classificacao_nps'] == 'Detrator') & (amostra['num_reclamacoes'] > 0) & (amostra['dias_atraso'] == 0)]
num_detratores_reclamacao_sem_atraso = detratores_reclamacao_sem_atraso.shape[0]
print(f"O número de Detratores que fizeram reclamação e não tiveram atraso na entrega é: {num_detratores_reclamacao_sem_atraso}")

Os detratores que não tiveram problemas na entrega é pequeno, e pode ser tratado por um atendimento personalizado para descobrir o que motivou isso.

In [ ]:
promotores_com_atraso = amostra[(amostra['classificacao_nps'] == 'Promotor') & (amostra['dias_atraso'] > 0)]
num_promotores_com_atraso = promotores_com_atraso.shape[0]
print(f"O número de Promotores que tiveram atraso na entrega é: {num_promotores_com_atraso}")

Existem clientes que não se incomodam com atrasos de entrega, mas é quase insignificante